In this lab, you will use the [Callbacks API](https://keras.io/api/callbacks/) to stop training when a specified metric is met. This is a useful feature so you won't need to complete all epochs when this threshold is reached. For example, if you set 1000 epochs and your desired accuracy is already reached at epoch 200, then the training will automatically stop. Let's see how this is implemented in the next section


## Load and Normalize the Fashion MNIST dataset

Like the previous lab, you will use the Fashion MNIST dataset again for this exercise. And also as mentioned before, you will normalize the pixel values to help optimize the training.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot

print(tf.__version__)

In [2]:
# Instantiate the dataset API
fmnist = tf.keras.datasets.fashion_mnist

# Load the dataset
(X_train,y_train),(X_test, y_test) = fmnist.load_data()


# Normalize the data
X_train = X_train / 255.0
X_test = X_test / 255.0


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Creating a Callback class

You can create a callback by defining a class that inherits the [tf.keras.callbacks.Callback](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/Callback) base class. From there, you can define available methods to set where the callback will be executed. For instance below, you will use the [on_epoch_end()](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/Callback#on_epoch_end) method to check the loss at each training epoch.

In [3]:
class MyCallBack(tf.keras.callbacks.Callback):
  def on_epoch_end(self,epoch, logs={}):
    '''
    Halts the traning after reaching 60 percent accuracy

    Args:
      epochs(integer) :-  index of epochs(required but unused in functions definition below)
      logs (dict) :- metric results from the traning epoch
    '''
    # Check accuracy
    if (logs.get('loss') < 0.4):
      # Stop if threshold is met
      print("\nLoss is lower than 0.4 so cancelling training!")
      self.model.stop_training = True

# Instantiate class
callbacks= MyCallBack()

## Define and compile the model
Next, you will define and compile the model. The architecture will be similar to the one you built in the previous lab. Afterwards, you will set the optimizer, loss, and metrics that you will use for training.

In [4]:
# Define the model
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Train the model

Now you are ready to train the model. To set the callback, simply set the callbacks parameter to the `myCallback` instance you declared before. Run the cell below and observe what happens.

In [6]:
# Train the model with a callback
model.fit(X_train, y_train, epochs=10, callbacks=[callbacks])

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8330 - loss: 0.4691
Epoch 2/10
1867/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8667 - loss: 0.3608
Loss is lower than 0.4 so cancelling training!
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.8686 - loss: 0.3578


You will notice that the training does not need to complete all 10 epochs. By having a callback at each end of the epoch, it is able to check the training parameters and compare if it meets the threshold you set in the function definition. In this case, it will simply stop when the loss falls below `0.40` after the current epoch.


###  Optional Challenge:
Modify the code to make the training stop when the accuracy metric exceeds 60%.

In [16]:
class AccuracyCallBack(tf.keras.callbacks.Callback):
  def on_epoch_end(self,epoch, logs={}):
    '''
    Halts the traning after reaching 60 percent accuracy

    Args:
      epochs(integer) :-  index of epochs(required but unused in functions definition below)
      logs (dict) :- metric results from the traning epoch
    '''
    # Check accuracy
    if (logs.get('accuracy') > 0.6):
      # Stop if threshold is met
      print("\nAccuracy  is greater  than 0.6 so cancelling training!")
      self.model.stop_training = True

# Instantiate class
Newcallbacks= AccuracyCallBack()

In [17]:
# Define the model
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])

# Compile the model
model.compile(optimizer=tf.optimizers.Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


# Train the model with a callback
model.fit(X_train, y_train, epochs=10, callbacks=[Newcallbacks])

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7936 - loss: 0.5872
Accuracy  is greater  than 0.6 so cancelling training!
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.8295 - loss: 0.4769
